# 04 — Evaluation across all three axes

1. **Guardrail**: zero-shot top-1 on CIFAR-100 (+ ImageNet val if mounted)
2. **Dense grounding**: zero-shot mIoU on PASCAL VOC
3. **Compositionality**: ARO / SugarCrepe / Winoground

Re-run after each fine-tuning checkpoint to track all three jointly.

In [ ]:
%cd /content/region-grounded
import torch
from peft import PeftModel
from transformers import AutoModel, AutoProcessor
from region_grounded import load_config

cfg = load_config('configs/default.yaml')
base = AutoModel.from_pretrained(cfg.stage3.base_model, torch_dtype=torch.float16).cuda().eval()
processor = AutoProcessor.from_pretrained(cfg.stage3.base_model)
# Optionally load LoRA adapters; if you want the baseline, skip this cell.
ADAPTER_PATH = f'outputs/{cfg.run_name}/final'
try:
    model = PeftModel.from_pretrained(base, ADAPTER_PATH).eval()
    print('loaded LoRA adapters from', ADAPTER_PATH)
except Exception as e:
    print('using base SigLIP (no adapters):', e)
    model = base

## Guardrail — CIFAR-100 zero-shot

In [ ]:
from region_grounded.eval_zeroshot import cifar100, zero_shot_top1
ds, classes = cifar100()
print(zero_shot_top1(model, processor, ds, classes))

## Grounding — PASCAL VOC mIoU

In [ ]:
import numpy as np
from torchvision.datasets import VOCSegmentation
from region_grounded.eval_segmentation import mean_iou

VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat',
    'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant',
    'sheep', 'sofa', 'train', 'tvmonitor',
]
voc = VOCSegmentation(root='data/voc', year='2012', image_set='val', download=True)
samples = ((img, np.array(mask)) for img, mask in voc)
print(mean_iou(model, processor, samples, VOC_CLASSES))

## Compositionality — ARO, SugarCrepe, Winoground

In [ ]:
from region_grounded.eval_compositionality import evaluate, aro_items, sugarcrepe_items, winoground_items
print('ARO  (VG_Relation):', evaluate(model, processor, aro_items('VG_Relation')))
print('SugarCrepe (replace_obj):', evaluate(model, processor, sugarcrepe_items('replace_obj')))
print('Winoground:', evaluate(model, processor, winoground_items()))